## Data Cleaning and Preprocessing

In this notebook we clean and preprocess all datasets before performing feature engineering and machine learning.

The cleaning process includes:

• Handling missing values  
• Fixing incorrect numerical values  
• Converting datatypes  
• Standardizing categorical values  
• Ensuring dataset relationships  
• Preparing datasets for feature engineering

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

## Load All Datasets

In [2]:
patient_df = pd.read_csv("../data/raw/1_patient_health_records.csv")
hospital_df = pd.read_csv("../data/raw/2_hospital_admissions.csv")
pharmacy_df = pd.read_csv("../data/raw/3_pharmacy_sales.csv")
prescription_df = pd.read_csv("../data/raw/4_doctor_prescriptions.csv")
weather_df = pd.read_csv("../data/raw/5_weather_data.csv")
population_df = pd.read_csv("../data/raw/6_population_demographics.csv")
outbreak_df = pd.read_csv("../data/raw/7_disease_outbreak.csv")
tweets_df = pd.read_csv("../data/raw/8_health_tweets.csv")
medicine_df = pd.read_csv("../data/raw/9_medicine_info.csv")

## Create Copies of DataFrames

We create copies to ensure original datasets remain unchanged.

In [3]:
patient = patient_df.copy()
hospital = hospital_df.copy()
pharmacy = pharmacy_df.copy()
prescription = prescription_df.copy()
weather = weather_df.copy()
population = population_df.copy()
outbreak = outbreak_df.copy()
tweets = tweets_df.copy()
medicine = medicine_df.copy()

## Fix Incorrect Numeric Values

### Fix Negative Weight Values

Weight cannot be negative.  
We convert negative weights to absolute values.

In [4]:

patient['weight_kg'] = patient['weight_kg'].abs()

### Fix BMI Values

BMI should not be negative.  
We recompute BMI using the correct formula.

In [5]:
height_m = patient['height_cm'] / 100

patient['bmi'] = patient['weight_kg'] / (height_m ** 2)

### Clip Physiological Values

Ensure realistic medical ranges.

In [6]:
patient['oxygen_level'] = patient['oxygen_level'].clip(70,100)
patient['blood_pressure_systolic'] = patient['blood_pressure_systolic'].clip(80,200)
patient['blood_pressure_diastolic'] = patient['blood_pressure_diastolic'].clip(50,130)
patient['heart_rate'] = patient['heart_rate'].clip(40,180)

## Handle Missing Values

### Handle Missing Diseases

In [34]:
patient['disease'] = patient['disease'].fillna("Healthy")

In [35]:
patient_disease_map = (
    patient.groupby('patient_id')['disease']
    .agg(lambda x: x.mode()[0])
)

hospital['disease'] = hospital['disease'].fillna(
    hospital['patient_id'].map(patient_disease_map)
)

hospital['disease'] = hospital['disease'].fillna("Unknown")

In [36]:
prescription['disease'] = prescription['disease'].fillna(
    prescription['patient_id'].map(patient_disease_map)
)

prescription['disease'] = prescription['disease'].fillna("Unknown")

### Handle Missing Alcohol Consumption

In [7]:
patient['alcohol_consumption'] = patient['alcohol_consumption'].fillna("Unknown")

### Fill Missing Disease in Hospital Dataset

We use the patient's known disease to fill missing hospital records.

In [ ]:
# Patient → Disease mapping
patient_disease_map = (
    patient.groupby('patient_id')['disease']
    .agg(lambda x: x.mode()[0] if not x.mode().empty else np.nan)
)

hospital['disease'] = hospital['disease'].fillna(
    hospital['patient_id'].map(patient_disease_map)
)

### Fill Missing Disease in Prescription Dataset

In [10]:
prescription['disease'] = prescription['disease'].fillna(
    prescription['patient_id'].map(patient_disease_map)
)

### Handle Missing Medicine Side Effects

In [11]:
medicine['side_effects'] = medicine['side_effects'].fillna("None")

## Convert Date Columns to Datetime

In [12]:
hospital['admission_date'] = pd.to_datetime(hospital['admission_date'])
hospital['discharge_date'] = pd.to_datetime(hospital['discharge_date'])

In [13]:
pharmacy['date'] = pd.to_datetime(pharmacy['date'])

In [14]:
prescription['date'] = pd.to_datetime(prescription['date'])

In [15]:
weather['date'] = pd.to_datetime(weather['date'])

In [16]:
outbreak['date'] = pd.to_datetime(outbreak['date'])

In [17]:
tweets['date'] = pd.to_datetime(tweets['date'])

## Remove Duplicate Rows

In [18]:
patient = patient.drop_duplicates()
hospital = hospital.drop_duplicates()
pharmacy = pharmacy.drop_duplicates()
prescription = prescription.drop_duplicates()
weather = weather.drop_duplicates()
population = population.drop_duplicates()
outbreak = outbreak.drop_duplicates()
tweets = tweets.drop_duplicates()
medicine = medicine.drop_duplicates()

## Standardize Categorical Columns

In [19]:
patient['gender'] = patient['gender'].str.title()

In [20]:
patient['smoking_status'] = patient['smoking_status'].str.title()

In [21]:
patient['physical_activity_level'] = patient['physical_activity_level'].str.title()

In [22]:
patient['disease'] = patient['disease'].str.title()
hospital['disease'] = hospital['disease'].str.title()
prescription['disease'] = prescription['disease'].str.title()
outbreak['disease'] = outbreak['disease'].str.title()

In [23]:
patient['city'] = patient['city'].str.title()
hospital['city'] = hospital['city'].str.title()
pharmacy['city'] = pharmacy['city'].str.title()
weather['city'] = weather['city'].str.title()
population['city'] = population['city'].str.title()
outbreak['city'] = outbreak['city'].str.title()
tweets['location'] = tweets['location'].str.title()

## Verify Data After Cleaning

In [37]:
print("Missing values after cleaning:\n")
print(patient.isnull().sum())

Missing values after cleaning:

patient_id                     0
age                            0
gender                         0
height_cm                      0
weight_kg                      0
bmi                            0
blood_pressure_systolic        0
blood_pressure_diastolic       0
cholesterol                    0
blood_glucose                  0
heart_rate                     0
oxygen_level                   0
smoking_status                 0
alcohol_consumption            0
physical_activity_level        0
sleep_hours                    0
family_history_diabetes        0
family_history_heart           0
city                           0
region                         0
disease                        0
symptom_fever                  0
symptom_cough                  0
symptom_fatigue                0
symptom_chest_pain             0
symptom_headache               0
symptom_shortness_of_breath    0
symptom_nausea                 0
symptom_joint_pain             0
dtype: int6

In [38]:
print("Missing values after cleaning:\n")
print(hospital.isnull().sum())

Missing values after cleaning:

admission_id      0
patient_id        0
hospital_id       0
city              0
admission_date    0
discharge_date    0
disease           0
severity_level    0
admission_type    0
treatment_type    0
icu_required      0
length_of_stay    0
treatment_cost    0
outcome           0
dtype: int64


In [27]:
print("Missing values after cleaning:\n")
print(pharmacy.isnull().sum())

Missing values after cleaning:

date                   0
city                   0
pharmacy_id            0
medicine_name          0
medicine_category      0
units_sold             0
price_per_unit         0
total_sales            0
prescriptions_count    0
temperature            0
humidity               0
rainfall               0
disease_cases          0
hospital_visits        0
dtype: int64


In [40]:
print("Missing values after cleaning:\n")
print(prescription.isnull().sum())

Missing values after cleaning:

prescription_id       0
doctor_id             0
hospital_id           0
patient_id            0
date                  0
disease               0
medicine_name         0
dosage                0
frequency             0
treatment_duration    0
prescription_type     0
dtype: int64


In [41]:
print("Missing values after cleaning:\n")
print(weather.isnull().sum())

Missing values after cleaning:

date                 0
city                 0
temperature          0
humidity             0
rainfall             0
wind_speed           0
air_quality_index    0
uv_index             0
season               0
dtype: int64


In [42]:
print("Missing values after cleaning:\n")
print(population.isnull().sum())

Missing values after cleaning:

city                     0
state                    0
population               0
population_density       0
median_age               0
male_population          0
female_population        0
urban_population         0
healthcare_facilities    0
hospital_beds            0
average_income           0
literacy_rate            0
dtype: int64


In [43]:
print("Missing values after cleaning:\n")
print(outbreak.isnull().sum())

Missing values after cleaning:

date                0
city                0
disease             0
cases_reported      0
hospitalizations    0
deaths              0
vaccination_rate    0
temperature         0
rainfall            0
humidity            0
dtype: int64


In [44]:
print("Missing values after cleaning:\n")
print(tweets.isnull().sum())

Missing values after cleaning:

tweet_id             0
tweet_text           0
location             0
date                 0
disease_mentioned    0
sentiment            0
retweets             0
likes                0
dtype: int64


In [45]:
print("Missing values after cleaning:\n")
print(medicine.isnull().sum())

Missing values after cleaning:

medicine_name            0
category                 0
primary_disease          0
dosage_form              0
dosage_strength          0
side_effects             0
average_price            0
manufacturer             0
prescription_required    0
dtype: int64


## Save Cleaned Datasets

In [46]:
patient.to_csv("../data/processed/patient_clean.csv", index=False)
hospital.to_csv("../data/processed/hospital_clean.csv", index=False)
pharmacy.to_csv("../data/processed/pharmacy_clean.csv", index=False)
prescription.to_csv("../data/processed/prescription_clean.csv", index=False)
weather.to_csv("../data/processed/weather_clean.csv", index=False)
population.to_csv("../data/processed/population_clean.csv", index=False)
outbreak.to_csv("../data/processed/outbreak_clean.csv", index=False)
tweets.to_csv("../data/processed/tweets_clean.csv", index=False)
medicine.to_csv("../data/processed/medicine_clean.csv", index=False)

## Data Cleaning Completed

All datasets are now:

• Free from major missing values  
• Standardized across categorical columns  
• Cleaned of incorrect numerical values  
• Converted to correct datatypes  

These cleaned datasets will now be used for:

1. Feature Engineering  
2. Exploratory Data Analysis  
3. Machine Learning  
4. NLP Modeling  
5. Deep Learning